In [1]:
# 什么是State

"""

state 使得不同工具可以读取或修改同一份数据

"""

from rich.console import Console

console = Console()

state = {

    "papers":[],
    "selected_ids": []

}

# 模拟搜索工具执行
state["papers"] = [
    {"id": "W1", "title": "Paper A", "citations": 100},
    {"id": "W2", "title": "Paper B", "citations": 50}
]

# 模拟排序工具执行
state["selected_ids"] = ["W1"]

console.print("搜索结果:", state["papers"])
console.print("选中论文:", state["selected_ids"])



搜索结果:
[{'id': 'W1', 'title': 'Paper A', 'citations': 100}, {'id': 'W2', 'title': 'Paper B', 'citations': 50}]

选中论文:
['W1']

### 如何让多个Tool 共享 State

In [2]:

# 两个 Tool 共享同一个 State

state = {"papers": [], "selected": []}


def search_papers():
    papers = [
        {"title": "Paper A", "citations": 50},
        {"title": "Paper B", "citations": 120},
        {"title": "Paper C", "citations": 80},
    ]

    # 将搜索结果保存到 State
    state["papers"] = papers

    return {"count": len(papers)}


def rank_papers(top_k: int = 2):
    # 直接从 State 中读取论文
    papers = state["papers"]

    ranked = sorted(
        papers,
        key=lambda p: p["citations"],
        reverse=True
    )[:top_k]

    state["selected"] = ranked

    return ranked


# 模拟 Agent 先后调用两个 Tool
print(search_papers())
print(rank_papers(top_k=2))


{'count': 3}
[{'title': 'Paper B', 'citations': 120}, {'title': 'Paper C', 'citations': 80}]


### 知识点 3：为什么要用 dataclass 管理 State

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AgentState:
    """
    
    这里需要理解两个语法。

    @dataclass：自动为类生成初始化方法等常用方法，让我们专注于定义数据结构。

    field(default_factory=list)：为每个 AgentState 实例创建独立的空列表，而不是多个实例共享同一个默认列表。
    
    """
    papers: list = field(default_factory=list)
    selected_ids: list = field(default_factory=list)
    last_keyword:str = ""

    def snapshot(self):

        return {
            "paper_count": len(self.papers),
            "selected_ids": self.selected_ids
        }

state = AgentState()

state.papers.append("Paper A")
state.selected_ids.append("W1")
state.last_keyword = "mechine learning"

console.print(state)
console.print(state.last_keyword)


AgentState(papers=['Paper A'], selected_ids=['W1'], last_keyword='mechine learning')

mechine learning

### 知识点 4：State 和 Context 有什么区别？

In [4]:
import json

"""

State:是 python 保存的全部数据

context:是实际传给模型的消息和信息

"""

# 模拟保存 100 篇文章
state = {
    "papers":[
        {
            "id": f"W{i}",
            "title": f"Paper {i}",
            "abstract": "Very long abstract" * 100
        }
        for i in range(100)
    ]
}

# State 中保存完整数据
full_data = json.dumps(state)

# 只取三篇论文的ID和标题给模型
context = {
    "paper_count": len(state["papers"]),
    "preview": [
        {"id": p["id"], "title": p["title"]}
        for p in state["papers"][:3]
    ]
}

context_data = json.dumps(context)

console.print("完整 State 字符数:", len(full_data))
console.print("简化 Context 字符数:", len(context_data))
console.print(context_data)





完整 State 字符数: 185192

简化 Context 字符数: 135

{"paper_count": 100, "preview": [{"id": "W0", "title": "Paper 0"}, {"id": "W1", "title": "Paper 1"}, {"id": "W2", 
"title": "Paper 2"}]}

### 六、知识点 5：State 与 Memory 有什么关系？

In [6]:
class MiniAgent:
    def __init__(self):
        self.state = {"selected_paper": None}

    def run(self, query: str):
        if query.startswith("选择"):
            self.state["selected_paper"] = query[2:]
            return "论文已选择"

        if "刚才" in query:
            return f"刚才选择的是：{self.state['selected_paper']}"

        return "没有相关操作"

agent = MiniAgent()

console.print(agent.run("选择Paper A"))
console.print(agent.run("刚才选择了什么论文?"))



论文已选择

刚才选择的是：Paper A

In [ ]:
import asyncio
from pydantic import BaseModel, Field
from app.agent.paper_search_agent import client, model


class TaskPlan(BaseModel):
    steps: list[str] = Field(min_length=1, max_length=6)


async def main():
    response = await client.responses.create(
        model=model,
        tools=[],
        input=[
            {
                "role": "system",
                "content": (
                    "将任务拆成最多6个简短待办步骤，只生成计划。"
                    '仅输出JSON：{"steps": ["步骤1", "步骤2"]}。'
                    "不要使用Markdown代码块。"
                ),
            },
            {
                "role": "user",
                "content": "搜索最多5篇雷达手势识别论文，选引用量最高的2篇，根据摘要比较方法。",
            },
        ],
    )

    plan = TaskPlan.model_validate_json(response.output_text)
    for i, step in enumerate(plan.steps, start=1):
        print(f"{i}. {step}")

await main()

1. 搜索最多5篇雷达手势识别论文
2. 记录各论文的标题、引用量和摘要
3. 按引用量排序并选出最高的2篇
4. 依据摘要提取两篇论文的方法要点
5. 比较两篇论文方法的异同并形成结论


In [ ]:
import asyncio
from rich.console import Console
from rich.markdown import Markdown
from app.agent.paper_search_agent import PaperSearchAgent, client, model

console = Console()

async def main():
    agent = PaperSearchAgent(
        client=client,
        model=model,
        max_steps=8,
    )

    answer = await agent.run(
        "搜索2023—2026年 radar gesture recognition 论文，最多5篇。"
        "按引用量选出前2篇，根据摘要比较识别方法，附标题和链接。"
        "如果选中的论文没有摘要，请明确说明。",
        target_papers=2,
    )
    console.print(Markdown(answer))


await main()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1694.56it/s]


本次任务计划

1. 搜索2023—2026年“radar gesture recognition”论文，最多保留5篇，并记录标题、链接、引用量与摘要状态。

2. 按引用量降序排序，选出引用量最高的前2篇。

3. 获取前2篇摘要；若某篇无摘要，明确标记“无摘要”。

4. 基于摘要提取并比较前2篇的识别方法。

... (14 outputs left out)

请求 OpenAlex 详情：W4402259598

摘要准备进度: {'target': 2, 'selected': 2, 'queried': 2, 'abstracts': 2, 'status': 'ready'}

第 5 轮模型调用

两篇论文的摘要都已经获取，下面汇总结果。

## 检索与处理进度说明
- **初始检索**：`radar gesture recognition`，2023—2026 年，返回 **5 篇**候选论文（上限为 5 篇）。
- **按引用量排序**：保留前 **2 篇**参与后续分析。
- **摘要可检索情况**：候选池 5 篇论文中，可检索到摘要的缓存数为 **0**（用英文检索词检索时，返回 0 条结果）。
- **摘要获取**：对选中的 2 篇论文逐篇调用详情接口获取摘要，**2 篇全部有摘要**，没有缺失。
- 以下“识别方法”描述均来自论文摘要中的明确表述，并非基于相关性推测。

## 前 2 篇论文（按引用量降序）

| 排名 | 标题 | 引用量 | 年份 | 链接 |
|---|---|---|---|---|
| 1 | Federated Learning for Radar Gesture Recognition Based on Spike Timing-Dependent Plasticity | 10 | 2024 | https://openalex.org/W4390806602 |
| 2 | Millimeter Wave Radar Gesture Recognition Based on ECC | 1 | 2024 | https://openalex.org/W4402259598 |

## 识别方法比较

**1. Federated Learning for Radar Gesture Recognition Based on Spike Timing-Dependent Plasticity（引用量 10）**

- **学习范式**：采用联邦学习框架。客户端之间只交换模型参数，不传输原始数据，以解决雷达信号处理中的隐私泄露和算力受限问题。
- **网络类型**：脉冲神经网络（SNN），使用**脉冲时序依赖可塑性（STDP）** 训练，具有生物可解释性，能显著降低能耗。
- **组合思路**：把低功耗 SNN 与联邦学习结合，缓解联邦学习中边缘节点的资源约束。
- **通信优化**：在联邦训练过程中引入**权重剪枝**，降低模型通信开销。
- **实验**：在 8GHz 雷达手势数据集和 Google Soli 数据